In [1]:
# Cell 0: Setup - ALWAYS run this first
%load_ext autoreload
%autoreload 2

from IPython.core.magic import register_cell_magic
@register_cell_magic
def skip(line, cell):
    return

import sys
from pathlib import Path
import os

# Add project root to path for src access
root = Path.cwd().parent
if str(root) not in sys.path:
    sys.path.append(str(root))

# Load environment variables
# from dotenv import load_dotenv
# load_dotenv()

# Verify setup
print(f"Project root: {root}")
print(f"Working directory: {Path.cwd()}")
print(f"Python path: {sys.path[:3]}...")

Project root: /Discs/E/D/Documents/Documents/Personal_Projects/JKproj
Working directory: /Discs/E/D/Documents/Documents/Personal_Projects/JKproj/notebooks
Python path: ['/home/stonecatter2/.local/share/uv/python/cpython-3.10.11-linux-x86_64-gnu/lib/python310.zip', '/home/stonecatter2/.local/share/uv/python/cpython-3.10.11-linux-x86_64-gnu/lib/python3.10', '/home/stonecatter2/.local/share/uv/python/cpython-3.10.11-linux-x86_64-gnu/lib/python3.10/lib-dynload']...


# Our 3 Datasets!

In [2]:
from datasets import load_dataset

# TCGA-UniformTumor-8K 

In [ ]:
# TCGA-UniformTumor-8K: ~200GB total (94 train shards × ~2GB each in parquet).
# ALWAYS use streaming=True — loading without it tries to download everything first.
uniform8k = load_dataset("MahmoodLab/TCGA-UniformTumor-8K", split="train", streaming=True)

# Peek at first sample to discover metadata fields (no full download needed)
sample = next(iter(uniform8k))
print("Fields:", list(sample.keys()))
print("\nNon-image metadata:")
for k, v in sample.items():
    if k != 'image':
        print(f"  {k}: {v}")

In [ ]:
# Filtering uniform8k (streaming) — filter() is lazy, iterates on-the-fly
# Example: keep only a specific cancer type (adjust field name after inspecting metadata above)
# filtered = uniform8k.filter(lambda x: x['cancer_type'] == 'LUAD')

# Example: filter by site/project and iterate
# for sample in filtered.take(5):
#     print(sample['cancer_type'], sample['site'])

# To collect metadata only (skip heavy image bytes) use select_columns if supported,
# or just access the non-image keys while iterating:
# for sample in uniform8k.take(100):
#     meta = {k: sample[k] for k in sample if k != 'image'}
#     print(meta)

# Dako

In [ ]:
# Dako (dakomura/tcga-ut): WebDataset/TAR format.
# name='external' = external validation cohort. Splits: train(192k), valid(39k), test(39k).
# Each tar shard is ~111MB — streaming still recommended to avoid downloading all at once.
dako = load_dataset('dakomura/tcga-ut', name='external', split='valid', streaming=True)

# Inspect a sample — key fields: '__key__', 'json', and image bytes
sample = next(iter(dako))
print("Fields:", list(sample.keys()))
print("__key__:", sample['__key__'])
print("json metadata:", sample['json'])

# TSS (Tissue Source Site) is encoded in __key__, e.g. 'TCGA-XX-XXXXXX/...'
tss = sample['__key__'].split('/')[0].split('-')[1]
print("TSS:", tss)
print("Label:", sample['json']['label'])

In [ ]:
from collections import defaultdict
import pandas as pd
from tqdm import tqdm

rows = []
for split in ['train', 'test', 'valid']:
    try:
        ds = load_dataset('dakomura/tcga-ut', name='external', split=split, streaming=True)
        for sample in tqdm(ds):
            tss = sample['__key__'].split('/')[0].split('-')[1]
            label = sample['json']['label']
            rows.append({'split': split, 'tss': tss, 'label': label})
    except Exception as e:
        print(f'{split}: {e}')

df = pd.DataFrame(rows)
summary = df.groupby(['tss', 'label', 'split']).size().reset_index(name='n')
summary.pivot_table(index=['tss','label'], columns='split', values='n', fill_value=0)


Resolving data files:   0%|          | 0/39 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/39 [00:00<?, ?it/s]

In [ ]:
# Filtering Dako (streaming)
# .filter() on a streaming dataset is lazy — it scans on-the-fly, no full download.

# Filter by label
tumor_only = dako.filter(lambda x: x['json']['label'] == 'tumor')

# Filter by TSS (Tissue Source Site from __key__)
def is_tss(sample, tss_code):
    parts = sample['__key__'].split('/')
    return parts[0].split('-')[1] == tss_code

site_filtered = dako.filter(lambda x: is_tss(x, 'A1'))  # replace 'A1' with target TSS

# Combine filters
combined = dako.filter(lambda x: x['json']['label'] == 'tumor' and is_tss(x, 'A1'))

# Iterate filtered stream (images only loaded on demand)
for sample in tumor_only.take(3):
    key  = sample['__key__']
    meta = sample['json']
    tss  = key.split('/')[0].split('-')[1]
    print(f"key={key}  tss={tss}  meta={meta}")